In [13]:
import torch 
from torch import nn
from torch import optim
from protgen.models.transformer import Transformer
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

Load data

In [14]:
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get train tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")

#Get validation tensors
X_val = val_data.tokens
y_val_labels = val_data.labels 
y_val_shifted = val_data.tokens_shifted

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])


Data Loaders

In [15]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

val_loader = DataLoader(val_data, batch_size=32, shuffle=False)


Loss function

In [16]:
transformer= Transformer()

cross_entropy_loss = nn.CrossEntropyLoss(ignore_index=-1) #ignore padding tokens

optimizer = optim.Adam(transformer.parameters(), lr=0.001)


Training loop

In [17]:
for epoch in range(20):
    transformer.train()

    train_loss_accumulated, num_batches, train_accuracy, num_tokens = 0.0, 0, 0, 0
    for x, x_shifted, _ , _ in train_loader:
        optimizer.zero_grad()
        logits= transformer(x) # (batch of proteins, max_seq_len, vocab_size)
        logits_reshaped= logits.reshape(-1, logits.size(-1)) # (batch of proteins * max_seq_len, vocab_size)

        targets= x_shifted.reshape(-1)
        loss= cross_entropy_loss(logits_reshaped, targets)

        loss.backward()

        optimizer.step()

        train_loss_accumulated += loss.item()
        num_batches += 1

        #necessary to exclude padding
        targets_masked = targets[targets != -1]
        logits_masked = logits_reshaped[targets != -1]
        train_next_token_accuracy = (logits_masked.argmax(dim=-1) == targets_masked).float().mean().item()
        train_accuracy += train_next_token_accuracy * targets_masked.size(0)
        num_tokens += targets_masked.size(0)

    train_loss_avg = train_loss_accumulated / num_batches
    train_accuracy_avg = train_accuracy / num_tokens
        

    transformer.eval()

    with torch.no_grad():
        val_loss_accumulated, num_batches, val_accuracy, num_tokens = 0.0, 0, 0, 0
        for x, x_shifted, _ , _ in val_loader:
            logits= transformer(x) # (batch of proteins, max_seq_len, vocab_size)
            logits_reshaped= logits.reshape(-1, logits.size(-1)) # (batch of proteins * max_seq_len, vocab_size)

            targets= x_shifted.reshape(-1)
            loss= cross_entropy_loss(logits_reshaped, targets)

            val_loss_accumulated += loss.item()
            num_batches += 1

            #same as train, necessary to exclude padding
            targets_masked = targets[targets != -1]
            logits_masked = logits_reshaped[targets != -1]
            val_next_token_accuracy = (logits_masked.argmax(dim=-1) == targets_masked).float().mean().item()
            val_accuracy += val_next_token_accuracy * targets_masked.size(0)
            num_tokens += targets_masked.size(0)

        val_loss_avg = val_loss_accumulated / num_batches
        val_accuracy_avg = val_accuracy / num_tokens

    print(f"Epoch {epoch+1}: Train Loss: {train_loss_avg:.4f}, Train Accuracy: {train_accuracy_avg:.4f}, Val Loss: {val_loss_avg:.4f}, Val Accuracy: {val_accuracy_avg:.4f}")
    # verify, if both decreases -> model is learning
    # if train loss decreases but val loss increases -> overfitting
    # if both losses are high and not decreasing -> underfitting
            
        


Epoch 1: Train Loss: 2.7953, Train Accuracy: 0.1374, Val Loss: 2.6793, Val Accuracy: 0.1783
Epoch 2: Train Loss: 2.6461, Train Accuracy: 0.1894, Val Loss: 2.5639, Val Accuracy: 0.2347
Epoch 3: Train Loss: 2.5702, Train Accuracy: 0.2240, Val Loss: 2.4934, Val Accuracy: 0.2617
Epoch 4: Train Loss: 2.5186, Train Accuracy: 0.2449, Val Loss: 2.4504, Val Accuracy: 0.2689
Epoch 5: Train Loss: 2.4855, Train Accuracy: 0.2554, Val Loss: 2.4208, Val Accuracy: 0.2811
Epoch 6: Train Loss: 2.4613, Train Accuracy: 0.2638, Val Loss: 2.4021, Val Accuracy: 0.2893
Epoch 7: Train Loss: 2.4437, Train Accuracy: 0.2693, Val Loss: 2.3829, Val Accuracy: 0.2947
Epoch 8: Train Loss: 2.4294, Train Accuracy: 0.2743, Val Loss: 2.3704, Val Accuracy: 0.2995
Epoch 9: Train Loss: 2.4186, Train Accuracy: 0.2782, Val Loss: 2.3622, Val Accuracy: 0.3002
Epoch 10: Train Loss: 2.4089, Train Accuracy: 0.2813, Val Loss: 2.3522, Val Accuracy: 0.3029
Epoch 11: Train Loss: 2.4009, Train Accuracy: 0.2837, Val Loss: 2.3489, Val Acc